# 📈 Stock Market — Next-Day Probability Model

**You do not need to install anything. You do not need to know GitHub.**

### How to run this

1. At the top of this page click **Runtime → Run all**
2. If a box pops up saying *"Warning: This notebook was not authored by Google"*, click **Run anyway**
3. Wait about **8–12 minutes**. Watch the ▶ symbols turn into ✔
4. Scroll down to read your results

That's it. Every step below explains what it's doing in plain English.

---

### What this model actually does

For each of ~120 big US stocks, it estimates:

> *"What is the probability that tomorrow this stock moves up by more than its own normal daily wiggle?"*

**Set your expectations before you start.** Predicting tomorrow's stock direction is
one of the hardest problems in finance. A **good** model here is right about
**52 out of 100 times**, not 90 out of 100. If you ever see a number that looks
amazing, that is a bug, not a discovery. This notebook is built to tell you the
difference honestly.

---
## Step 1 — Set up (about 1 minute)

Downloads the code and the tools it needs. You will see a lot of text scroll by. That's normal.

In [ ]:
# Download the project and install the tools it needs.
import os, sys, subprocess

if not os.path.exists('/content/quant-lab'):
    subprocess.run(
        ['git', 'clone', '--quiet',
         'https://github.com/bader7375/quant-lab.git', '/content/quant-lab'],
        check=True)

os.chdir('/content/quant-lab')
sys.path.insert(0, '/content/quant-lab/src')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'yfinance', 'lightgbm', 'pyarrow', 'tabulate',
                'pandas>=2.1', 'scikit-learn>=1.4'], check=True)

import pandas as pd
print(f"pandas {pd.__version__}  (needs 2.1 or newer)")

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s',
                    datefmt='%H:%M:%S', force=True)

print("\n✅ Setup finished. The code is ready to run.")

---
## Step 2 — Self-test (about 1 minute)

Before touching real market data, the code checks **itself** for bugs.

The most dangerous bug in this kind of project is called **"lookahead"** — where the
model accidentally peeks at tomorrow's answer while making today's prediction. That
makes a broken model look brilliant. These 35 tests exist to catch exactly that.

**You want to see `35 passed`.** If you don't, stop and tell Claude.

In [ ]:
# Run the safety tests. This uses fake, simulated market data - no internet needed.
!cd /content/quant-lab && PYTHONPATH=src python -m pytest tests/ -q 2>&1 | tail -5

---
## Step 3 — Download real market data and train (about 5–10 minutes)

Now the real thing. This will:

1. **Download** ~120 large US stocks (Apple, Microsoft, etc.) from Yahoo Finance, back to 2015
2. **Build 108 clues** per stock per day — recent returns, volatility, trading volume,
   how it ranks against every other stock that day, and so on
3. **Train and test 4 times**, always training on the *past* and testing on the *future*,
   never the other way around

☕ **This is the slow part.** Let it run. Progress messages appear as it goes.

**If it fails:** you'll get a tidy diagnostic report instead of a scary traceback.
Copy the whole thing into the chat and Claude can fix it. The most common cause is
Yahoo Finance temporarily rate-limiting Google's servers — waiting 15 minutes and
re-running often fixes it on its own.

In [ ]:
from quantlab.config import Config
from quantlab import pipeline

# ---- Settings you can change ----------------------------------------------
cfg = Config.load(None, **{
    'data.provider': 'yfinance',   # real market data from Yahoo Finance
    'data.universe': 'sp100',      # ~120 big US stocks. Use 'sp500' for all 500 (slower)
    'data.start':    '2015-01-01', # how far back to look. Earlier = better, but slower
    'split.n_folds':  4,           # how many times to train-and-test
    'split.min_train_days': 750,   # ~3 years of history before the first prediction
    'output_dir': 'artifacts/colab',
})
# ---------------------------------------------------------------------------

try:
    results = pipeline.run(cfg)
    print("\n✅ Done. Scroll down for your results.")
except Exception as e:
    # Don't leave you staring at a raw Python traceback. Print something you
    # can paste straight into the chat instead.
    from quantlab.diagnostics import preflight
    print("\n\n")
    print(preflight(e))

---
## Step 4 — Your results, in plain English

The cell below translates the numbers into sentences. Read this before the charts.

In [ ]:
r, c, b = results['ranking'], results['classification'], results['backtest']

def verdict(auc, t):
    if auc < 0.505 or t < 2:   return "❌ NO REAL SIGNAL", "The model did not beat random guessing. This is the most common outcome, and it is an honest result."
    if auc < 0.52:             return "🟡 WEAK / BORDERLINE", "There may be something here, but it is faint. Do not trade on this."
    if auc < 0.57:             return "🟢 REAL SIGNAL", "This is a genuine, research-grade daily signal. Now check whether it survives trading costs."
    return "🚨 TOO GOOD - SUSPECT A BUG", "Scores this high on next-day prediction almost always mean data leaked from the future. Investigate before believing it."

tag, note = verdict(r['daily_auc_mean'], r['daily_auc_t_stat'])

print("=" * 68)
print("  IS THE MODEL ANY GOOD AT PICKING STOCKS?")
print("=" * 68)
print(f"  Score (AUC)      : {r['daily_auc_mean']:.4f}   (0.50 = coin flip, 0.55 = excellent)")
print(f"  Confidence       : t = {r['daily_auc_t_stat']:.1f}     (above 3 = probably not luck)")
print(f"  Days tested      : {r['ic_days']:,.0f}")
print()
print(f"  VERDICT: {tag}")
print(f"  {note}")
print()
print("=" * 68)
print("  WOULD IT ACTUALLY MAKE MONEY?")
print("=" * 68)
print(f"  Profit before trading fees : {b['gross_ann_return']*100:>7.1f}% per year")
print(f"  Profit after trading fees  : {b['net_ann_return']*100:>7.1f}% per year")
print(f"  Worst loss from a peak     : {b['max_drawdown']*100:>7.1f}%")
print()
print(f"  ⭐ BREAK-EVEN TRADING COST : {b['breakeven_cost_bps']:.1f} basis points")
print(f"     If it costs you MORE than {b['breakeven_cost_bps']:.1f} bps to trade, you lose money.")
print(f"     A realistic cost for a normal person is 5-10 bps.")
if b['breakeven_cost_bps'] < 5:
    print("     ➜ Below 5. This signal is NOT tradeable in real life.")
elif b['breakeven_cost_bps'] < 10:
    print("     ➜ Marginal. Tradeable only with very cheap execution.")
else:
    print("     ➜ Comfortably tradeable, if the signal holds up out of sample.")
print("=" * 68)

---
## Step 5 — The charts

Six pictures. Here is what to look for in each:

| Chart | What it shows | What "good" looks like |
|---|---|---|
| **Equity curve** | Money made over time | Both lines rising. **The orange line is the one that matters** — that's after fees |
| **Reliability** | When it says "60%", does it happen 60% of the time? | Dots close to the dashed line |
| **Return by decile** | Stocks it liked most vs least | A rising staircase, left to right |
| **Information coefficient** | Its skill over time | Mostly above zero, not just one lucky year |
| **Drawdown** | Losing streaks | Shallow. Deep valleys = painful to live through |
| **Top features** | Which clues it relied on | See the warning below |

⚠️ **On that last chart:** if the top clues all start with `mkt_` or `breadth_`, the model
is guessing the *whole market's* direction rather than picking individual stocks.

In [ ]:
from IPython.display import Image, display
display(Image(filename='artifacts/colab/report.png'))

---
## Step 6 — Did it work in every period, or just get lucky once?

A model that only worked in one stretch of time has found nothing durable. You want the
`auc` column to be above 0.50 in **most** rows, not just one.

`base_rate` is what you'd get by always guessing the same answer — **always compare
`accuracy` to `base_rate`**, never to 50%.

In [ ]:
import pandas as pd
folds = pd.read_csv('artifacts/colab/fold_metrics.csv')
display(folds[['fold', 'test_start', 'test_end', 'auc', 'accuracy', 'base_rate']].round(4))

print("\nHow much profit survives at different trading costs:")
display(pd.read_csv('artifacts/colab/cost_sensitivity.csv').round(3))

---
## Step 7 — Tomorrow's probabilities

Finally, the thing you asked for: the model retrains on **all** the history available, then
scores every stock for the next trading day.

**How to read the table:**
- `probability` — chance this stock rises more than its normal daily move. **These will be
  bunched together in a narrow band, usually near 0.50, and that is correct.** The model is
  genuinely uncertain about any single stock. A model claiming 80% on one stock tomorrow is
  lying to you. You may also see several stocks share the exact same probability — that just
  means the model cannot tell them apart.
- `cs_rank` — **this is the useful column.** 0.99 means "of all stocks today, this one looks
  best." Ranking is where the real information is, not the raw percentage.
- `threshold_move` — how big a move counts as "up" for this particular stock.

In [ ]:
preds = pipeline.predict_latest(cfg)

print(f"\nMost promising stocks for the next trading day:\n")
display(preds[['probability', 'cs_rank', 'threshold_move']].head(15).round(4))

print("\nLeast promising:\n")
display(preds[['probability', 'cs_rank', 'threshold_move']].tail(10).round(4))

preds.to_csv('tomorrow_predictions.csv')
print("\n💾 Saved to tomorrow_predictions.csv (left panel → Files → download)")

---
## What to try next

Go back to **Step 3**, change one setting, then **Runtime → Run all** again.

| Change this | To this | What happens |
|---|---|---|
| `'data.universe'` | `'sp500'` | All 500 stocks instead of 120. Slower, usually better — more data to learn from |
| `'data.start'` | `'2005-01-01'` | 20 years of history including the 2008 crash. Slower, usually better |
| `'split.n_folds'` | `8` | Tests over more separate time periods — a stricter check |
| `'label.threshold_sigma'` | `0.5` | Only predicts *bigger* moves. Fewer signals, but usually cleaner ones |
| `'backtest.cost_bps'` | `2.0` | Assumes cheaper trading. Realistic only for professionals |

### An experiment worth doing once

Add `'data.provider': 'synthetic'` and `'data.synthetic_signal_strength': 1.0`.

That runs on a **simulated** market where a real signal was deliberately hidden in the data —
at the strength you'd actually find in real markets. Watch how nearly invisible it is even
though you *know* it's there. That single run will teach you more about why this problem is
hard than any amount of reading.

### The honest caveat

The stock list is **today's** S&P 500, applied backwards through history. Companies that went
bankrupt or got delisted are missing, so the model only ever sees survivors. This makes results
look better than reality. It's the biggest known flaw, it's documented in the project README,
and fixing it needs a historical membership list.

**Nothing here is financial advice.** This is a research tool for understanding how hard the
problem is — treat any result as a hypothesis to investigate, not a reason to trade.